In [35]:
import csv, pandas as pd 
import psycopg2
from psycopg2.extras import execute_values
import numpy as np
import io
from datetime import datetime
from uuid import UUID
import random
import re
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv
import os

In [36]:
load_dotenv()
env = os.environ

SSH_HOST = env.get("SSH_HOST")
SSH_PORT = int(env.get("SSH_PORT", 22))
SSH_USERNAME = env.get("SSH_USERNAME")
SSH_PASSWORD = env.get("SSH_PASSWORD")
SSH_PRIVATE_KEY = env.get("SSH_PRIVATE_KEY")

DB_HOST = env.get("DB_HOST")
DB_PORT = int(env.get("DB_PORT", 5432))
DB_NAME = env.get("DB_NAME")
DB_USER = env.get("DB_USER")
DB_PASSWORD = env.get("DB_PASSWORD")

# ======================================================
# Create SSH Tunnel
# ======================================================
tunnel_args = {
    "ssh_address_or_host": (SSH_HOST, SSH_PORT),
    "ssh_username": SSH_USERNAME,
    "remote_bind_address": (DB_HOST, DB_PORT),
    "allow_agent": False,
    "host_pkey_directories": [],
}

if SSH_PASSWORD:
    tunnel_args["ssh_password"] = SSH_PASSWORD

if SSH_PRIVATE_KEY and os.path.isfile(SSH_PRIVATE_KEY):
    tunnel_args["ssh_pkey"] = SSH_PRIVATE_KEY

conn = None
try:
    tunnel = SSHTunnelForwarder(**tunnel_args)
    tunnel.start()
    print(f"✓ SSH Tunnel Established (localhost:{tunnel.local_bind_port})")

    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host="127.0.0.1",
        port=tunnel.local_bind_port,
    )
    print("✓ Database connected successfully!")
except Exception as e:
    print(f"Connection warning/error: {e}")

✓ SSH Tunnel Established (localhost:35645)
✓ Database connected successfully!


In [37]:
kmat = pd.read_csv('KMAT.csv', delimiter="\t")
kmat

,Old material no.,Material description,Language,New Material
0,001A,LT-25 - FANUC,EN,40001
1,001B,LT-25 - SIEMENS,EN,40002
2,002A,LT-25 630 - FANUC,EN,40003
3,003A,LT-20 C CLASSIC - FANUC,EN,40004
4,003B,LT-20 C CLASSIC - SIEMENS,EN,40005
...,...,...,...,...
568,SUPERWINNER,SUPERWINNER,EN,40516
569,TCV-540,TCV540,EN,40517
570,WINNER,WINNER,EN,40518
571,M30,MCV450 PLUS,EN,40553


In [38]:
with open('event_log_master_27Aug2026.csv', 'r', encoding='utf-8', errors='replace') as f:
    content = f.read().replace('Sector" B"', 'Sector B')

df = pd.read_csv(
    io.StringIO(content),
    delimiter=",",
    engine="python"
)

print("Total records in df : ", len(df))

Total records in df :  58


In [39]:
# 1. segregate out Cancelled Machines
cancelled_df = df[df['cancel_indicator'] == 'X']
cancelled_df.to_csv('01_dq_cancelled_machines.csv', index=False, quoting=csv.QUOTE_ALL)

df = df[df['cancel_indicator'].isna()].reset_index(drop=True)
print("Total records : ", len(df))

Total records :  58


In [40]:
# 2. find duplicates of cnc id machines
duplicates_cnc = df[df.duplicated(subset=['cnc_id'], keep=False) & df['cnc_id'].notna()]
duplicates_cnc.to_csv('02_dq_duplicated_cnc.csv', index=False, quoting=csv.QUOTE_ALL)

null_cnc = df[df['cnc_id'].isnull()]
null_cnc.to_csv('03_dq_null_cnc.csv', index=False, quoting=csv.QUOTE_ALL)

# Keep one record for each CNC ID
df = (
    df[df['cnc_id'].notna()]
    .drop_duplicates(subset=['cnc_id'], keep='first')
    .reset_index(drop=True)
)

print("Total records : ", len(df))

Total records :  28


In [41]:
# 3. Invalid CNCID Pattern
pattern = r'^[^-]{8}(?:-[^-]{8}){3}$'

invalid_cnc_id = df[~df["cnc_id"].astype(str).str.match(pattern, na=False)]

df = df[
    df["cnc_id"].astype(str).str.match(pattern, na=False)
].copy()

invalid_cnc_id.to_csv('04_dq_invalid_cncid_pattern.csv', sep=',', quoting=csv.QUOTE_ALL, index=False)

print("Total records : ", len(df))

Total records :  28


In [ ]:
# # 3b. Check that cnc_id is already present in sap.fact_machine_dispatch_logs
# if conn is not None:
#     query = """
#     SELECT DISTINCT cnc_id 
#     FROM sap.fact_machine_dispatch_logs 
#     WHERE cnc_id IS NOT NULL;
#     """
#     with conn.cursor() as cur:
#         cur.execute(query)
#         existing_cnc_ids = set(str(row[0]).strip() for row in cur.fetchall() if row[0] is not None)

#     already_present_cnc_id = df[df["cnc_id"].astype(str).str.strip().isin(existing_cnc_ids)]
#     df = df[~df["cnc_id"].astype(str).str.strip().isin(existing_cnc_ids)].copy()
#     already_present_cnc_id.to_csv('04b_dq_already_present_cncid.csv', sep=',', quoting=csv.QUOTE_ALL, index=False)
#     print("Total records after removing existing DB CNC IDs: ", len(df))

In [42]:
# 4. Update Supplier 'ACE' -> 'MCD' and 'AMS' -> 'TCD'
df['supplier'] = df['supplier'].replace({
    'ACE': 'MCD',
    'AMS': 'TCD'
})

In [43]:
original_df = df.copy()

# ------------------------------------------------------------------
# 1. Clean input columns
# ------------------------------------------------------------------
df['supplier_product'] = df['supplier_product'].astype(str).str.strip()
df['customer_product_slno'] = df['customer_product_slno'].astype(str).str.strip()

# ------------------------------------------------------------------
# 2. KMAT lookup tables
# ------------------------------------------------------------------
kmat_desc_lookup = kmat[
    ['Material description', 'Old material no.', 'New Material']
].drop_duplicates(subset=['Material description'])

kmat_newmat_lookup = kmat[
    ['Material description', 'New Material', 'Old material no.']
].drop_duplicates(subset=['New Material'])

# ------------------------------------------------------------------
# 3. FIRST: Direct match on Material Description using supplier_product
# ------------------------------------------------------------------
direct_df = df.merge(
    kmat_desc_lookup,
    left_on='supplier_product',
    right_on='Material description',
    how='inner'
)

direct_df['MappingType'] = 0

# ------------------------------------------------------------------
# 4. Keep only unmatched rows
# ------------------------------------------------------------------
remaining_df = df[
    ~df['supplier_product'].isin(direct_df['supplier_product'])
].copy()

# ------------------------------------------------------------------
# 5. Fallback logic on unmatched rows
# ------------------------------------------------------------------
kmat_descriptions = set(
    kmat['Material description'].dropna().astype(str).str.strip()
)
kmat_new_materials = set(
    kmat['New Material'].dropna().astype(str).str.strip()
)

supplier_prod = remaining_df['supplier_product'].astype(str).str.strip()
customer_slno = remaining_df['customer_product_slno'].astype(str).str.strip()

has_hyphen = customer_slno.str.contains('-', na=False)
slno_prefix = customer_slno.str.split('-').str[0].str.strip()

match_description = supplier_prod.isin(kmat_descriptions)
match_new_material = has_hyphen & slno_prefix.isin(kmat_new_materials)

# Keep only rows matching KMAT
remaining_df = remaining_df[
    match_description | match_new_material
].reset_index(drop=True)

# Prepare keys
remaining_df['supplier_product'] = remaining_df['supplier_product'].astype(str).str.strip()
remaining_df['customer_product_slno'] = remaining_df['customer_product_slno'].astype(str).str.strip()
remaining_df['Prefix'] = remaining_df['customer_product_slno'].str.split('-').str[0]

# Split into two groups
desc_df = remaining_df[
    ~remaining_df['customer_product_slno'].str.contains('-', na=False)
].copy()

newmat_df = remaining_df[
    remaining_df['customer_product_slno'].str.contains('-', na=False)
].copy()

# Merge by Material description on supplier_product
desc_df = desc_df.merge(
    kmat_desc_lookup,
    left_on='supplier_product',
    right_on='Material description',
    how='left',
)

# Merge by New Material on Prefix
newmat_df = newmat_df.merge(
    kmat_newmat_lookup,
    left_on='Prefix',
    right_on='New Material',
    how='left'
)

# Mapping type indicator
desc_df['MappingType'] = 0
newmat_df['MappingType'] = 1

# ------------------------------------------------------------------
# 6. Final result
# ------------------------------------------------------------------
df = pd.concat(
    [direct_df, desc_df, newmat_df],
    ignore_index=True
)

df.drop(columns=['Prefix'], inplace=True, errors='ignore')

# Use official KMAT Material description to update supplier_product
df['supplier_product'] = df['Material description'].fillna(df['supplier_product'])

print("Total records:", len(df))

Total records: 26


In [44]:
dummy_df = original_df[
    ~original_df["cnc_id"].isin(df["cnc_id"])
].copy()

dummy_df.to_csv("05_dq_out_of_kmat_cnc.csv", index=False, sep=",", quoting=csv.QUOTE_ALL)

# Free memory
del original_df

In [45]:
# 6. feature hardcoded
df['features'] = 10000000

In [46]:
# 7. updated Invoice and InvoiceItem with random Values
df['invoice'] = df['invoice'].astype('string')
df['invoice_item'] = df['invoice_item'].astype('string')
used = set(df['invoice'].dropna().astype(str))

def random_invoice():
    while True:
        inv = str(random.randint(100, 999))
        if inv not in used:
            used.add(inv)
            return inv

invoice_updates = {}
invoiceitem_updates = {}

for (customer, cncid), grp in df.groupby(['customer', 'cnc_id'], sort=False):

    # -------- Invoice --------
    inv = None

    # Reuse existing invoice if present in this CNC group
    existing = grp['invoice'].dropna()
    if not existing.empty:
        inv = str(existing.iloc[0])
    else:
        inv = random_invoice()

    # Fill only missing invoices
    for idx in grp[grp['invoice'].isna()].index:
        invoice_updates[idx] = inv

    # -------- InvoiceItem --------
    item_rows = grp[grp['invoice_item'].isna()].index

    for i, idx in enumerate(item_rows, start=1):
        invoiceitem_updates[idx] = str(i * 10)

# Bulk update
df.loc[invoice_updates.keys(), 'invoice'] = list(invoice_updates.values())
df.loc[invoiceitem_updates.keys(), 'invoice_item'] = list(invoiceitem_updates.values())

In [47]:
# 8. update sap.dim_product table first
now = datetime.now()
zero_uuid = "00000000-0000-0000-0000-000000000000"

upsert_df = (
        df[
            [
                "supplier",
                "Material description",
                "Old material no.",
                "New Material",
            ]
        ]
        .dropna(subset=["Material description"])
        .copy()
        .rename(
            columns={
                "Material description": "supplier_product",
                "Old material no.": "old_material_no",
                "New Material": "new_material_no",
            }
        )
    )

upsert_df["supplier_id"] = upsert_df["supplier"].apply(
        lambda x: 1 if str(x).strip().upper() == "MCD" else 2
    )

upsert_df = upsert_df.drop_duplicates(
        subset=["supplier_id", "supplier_product"]
    )

for col in ["supplier_product", "old_material_no", "new_material_no"]:
        upsert_df[col] = (
            upsert_df[col]
            .astype(str)
            .str.strip()
            .replace({"nan": None, "None": None, "": None})
        )

upsert_df["created_by"] = zero_uuid
upsert_df["updated_by"] = zero_uuid
upsert_df["created_at"] = now
upsert_df["updated_at"] = now
upsert_df["updated_type"] = "service"

upsert_df = upsert_df[
        [
            "supplier_id",
            "supplier_product",
            "created_by",
            "updated_by",
            "created_at",
            "updated_at",
            "updated_type",
            "old_material_no",
            "new_material_no",
        ]
    ]

upsert_df = upsert_df.where(upsert_df.notna(), None)
records = list(upsert_df.itertuples(index=False, name=None))

In [ ]:
sql = """
    INSERT INTO sap.dim_product (
        supplier_id,
        supplier_product,
        created_by,
        updated_by,
        created_at,
        updated_at,
        updated_type,
        old_material_no,
        new_material_no
    )
    VALUES %s
    ON CONFLICT (supplier_id,supplier_product)
    DO UPDATE SET
        supplier_id     = EXCLUDED.supplier_id,
        old_material_no = EXCLUDED.old_material_no,
        new_material_no = EXCLUDED.new_material_no,
        updated_by      = EXCLUDED.updated_by,
        updated_at      = EXCLUDED.updated_at,
        updated_type    = EXCLUDED.updated_type;
    """
if conn is not None:
    with conn.cursor() as cur:
        execute_values(cur, sql, records, page_size=1000)

    conn.commit()
    print(f"Upserted {len(records)} records into sap.dim_product")

In [48]:
# 9. Format Bronze layer DataFrame
brz_df = df.copy()

brz_df["job_id"] = 23
brz_df["ingested_ts"] = pd.Timestamp.now()
brz_df["created_at"] = pd.Timestamp.now()

# Convert dispatch_date
brz_df["dispatch_date"] = pd.to_datetime(
    brz_df["dispatch_date"], errors="coerce"
).dt.date

db_columns = [
    "job_id",
    "branch",
    "location",
    "customer",
    "supplier",
    "supplier_product",
    "customer_product_slno",
    "dispatch_date",
    "commissioned_date",
    "customer_product_status",
    "features",
    "cnc_id",
    "updated_by",
    "updated_ts",
    "invoice",
    "cancel_indicator",
    "invoice_item",
    "customer_no",
    "customer_address",
    "customer_city",
    "customer_pincode",
    "ship_to_no",
    "ship_to_name",
    "ship_to_address",
    "ship_to_city",
    "ship_to_pincode",
    "ingested_ts",
    "created_at",
]

brz_df = brz_df[db_columns]
brz_df = brz_df.replace(r"^\s*$", np.nan, regex=True)
brz_df = brz_df.where(pd.notnull(brz_df), None)

print("brz_df shape:", brz_df.shape)
print("Sample supplier_product & customer_product_slno values:")
print(brz_df[["supplier_product", "customer_product_slno", "job_id"]].head())

brz_df shape: (26, 28)
Sample supplier_product & customer_product_slno values:
       supplier_product customer_product_slno  job_id
0      J 300 LM - FANUC           40209-14103      23
1      JE 06 LM - FANUC            40244-5725      23
2      JE 06 LM - FANUC            40244-5726      23
3  VTL-300 XL L - FANUC               40384-6      23
4                  430V             40469-714      23


In [49]:
brz_df.to_csv('bronze_df_20260825.csv', index=False, quoting=csv.QUOTE_ALL)

In [50]:
# 10. Execute Bulk COPY into bronze.raw_machine_dispatch_logs
if conn is not None:
    buffer = io.StringIO()
    brz_df.to_csv(buffer, index=False, header=False, sep='\t', na_rep='\\N')
    buffer.seek(0)

    copy_sql = r"""
    COPY bronze.raw_machine_dispatch_logs (
        job_id,
        branch,
        location,
        customer,
        supplier,
        supplier_product,
        customer_product_slno,
        dispatch_date,
        commissioned_date,
        customer_product_status,
        features,
        cnc_id,
        updated_by,
        updated_ts,
        invoice,
        cancel_indicator,
        invoice_item,
        customer_no,
        customer_address,
        customer_city,
        customer_pincode,
        ship_to_no,
        ship_to_name,
        ship_to_address,
        ship_to_city,
        ship_to_pincode,
        ingested_ts,
        created_at
    ) FROM STDIN WITH (FORMAT csv, DELIMITER E'\t', NULL '\N');
    """

    with conn.cursor() as cur:
        cur.copy_expert(sql=copy_sql, file=buffer)

    conn.commit()
    print("Bulk COPY inserted successfully into bronze.raw_machine_dispatch_logs!")

Bulk COPY inserted successfully into bronze.raw_machine_dispatch_logs!


In [ ]:
# 11. Update sap.dim_branch
if conn is not None:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT branch_name, location_name
            FROM sap.dim_branch
        """)
        db_combos = {
            (branch.strip(), location.strip())
            for branch, location in cur.fetchall()
        }

    branch_df = (
        df[['branch', 'location']]
        .dropna(subset=['branch', 'location'])
        .copy()
    )

    branch_df["branch"] = branch_df["branch"].astype(str).str.strip()
    branch_df["location"] = branch_df["location"].astype(str).str.strip()
    branch_df["region"] = None

    branch_df = branch_df.drop_duplicates(subset=["branch", "location"])

    missing_df = branch_df[
        ~pd.MultiIndex.from_frame(
            branch_df[["branch", "location"]]
        ).isin(db_combos)
    ].copy()

    if not missing_df.empty:
        now = datetime.now()
        zero_uuid = "00000000-0000-0000-0000-000000000000"
        missing_df["created_by"] = zero_uuid
        missing_df["updated_by"] = zero_uuid
        missing_df["created_at"] = now
        missing_df["updated_at"] = now
        missing_df["updated_type"] = "service"

        records = list(
            missing_df[
                [
                    "branch",
                    "location",
                    "created_by",
                    "updated_by",
                    "created_at",
                    "updated_at",
                    "region",
                    "updated_type",
                ]
            ].itertuples(index=False, name=None)
        )

        sql_branch = """
        INSERT INTO sap.dim_branch (
            branch_name,
            location_name,
            created_by,
            updated_by,
            created_at,
            updated_at,
            region,
            updated_type
        )
        VALUES %s
        ON CONFLICT (branch_name, location_name) DO NOTHING;
        """

        with conn.cursor() as cur:
            execute_values(cur, sql_branch, records, page_size=1000)

        conn.commit()
        print(f"Inserted {len(records)} new branch/location combinations into sap.dim_branch.")